# Baseline scenario: LightGBM

`scenario_id = baseline_lightgbm`

LightGBM (300 trees) per (target, horizon) cell, on the *identical*
baseline feature set as `04_baseline_linear_regression.ipynb` —
seasonality, lags, and (load only) weekday features. No weather or
capacity information yet. See `docs/modeling_plan.md` for the full plan
and `dhf/features.py` for the feature definitions.

This notebook follows the same structure as the linear regression one
(features used -> MAE/RMSE -> actual-vs-predicted -> error analysis ->
feature importance) so the two are easy to compare section by section;
the shared computation/plotting code lives in `dhf/reporting.py`.

In [1]:
import pandas as pd
from lightgbm import LGBMRegressor

from dhf.paths import ProjPaths
from dhf.reporting import (
    describe_feature_sets,
    fit_and_evaluate,
    plot_actual_vs_predicted,
    plot_importance_heatmaps,
    plot_mae_by_horizon,
    plot_rolling_error,
)
from dhf.scenarios import save_scenario_metrics
from dhf.splits import EXAMPLE_WINDOWS
from dhf.viz import KNOWN_SCENARIO_COLORS, MODEL_LABELS

paths = ProjPaths()
paths.ensure_directories()

SCENARIO_ID = "baseline_lightgbm"
COLOR = KNOWN_SCENARIO_COLORS[SCENARIO_ID]
MODEL_LABEL = MODEL_LABELS["lightgbm"]

panel = pd.read_parquet(paths.target_panel_file)

## Features used

Feature columns are identical across all three horizons for a given
target (only their *values* change — see `dhf/features.py`), so it's
enough to list them once per target.

In [2]:
for target, features in describe_feature_sets(panel).items():
    print(f"{target:15s} ({len(features)} features): {', '.join(features)}")

pv              (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
wind_onshore    (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
wind_offshore   (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
load            (15 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h, lag_weekday_match, weekday_Tuesday, weekday_Wednesday, weekday_Thursday, weekday_Friday, weekday_Saturday, weekday_Sunday


## Fit and evaluate

One `LGBMRegressor` per (target, horizon) cell, trained on data before
the validation cutoff and scored on the validation period only (test is
reserved — see `docs/modeling_plan.md`).

In [3]:
def make_model():
    return LGBMRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        importance_type="gain",
    )


def importance_fn(model, feature_names):
    return pd.Series(model.feature_importances_, index=feature_names)


results, val_predictions, importances = fit_and_evaluate(panel, make_model, importance_fn)

save_scenario_metrics(
    paths,
    scenario_id=SCENARIO_ID,
    title=MODEL_LABEL,
    description=(
        "LightGBM (300 trees, max_depth=6) on the identical feature set as the linear "
        "regression baseline. No weather/capacity data."
    ),
    notebook="05_baseline_lightgbm",
    valid=True,
    results=results,
)
print(f"Saved -> {paths.scenario_results_file(SCENARIO_ID)}")

Saved -> /home/chris/research/delu-headline-forecast/data/processed/results/baseline_lightgbm.parquet


## Results overview

MAE is the primary metric; RMSE is reported alongside for context.

In [4]:
overview = results.set_index(["target", "horizon_days"])[["mae", "rmse", "n_train", "n_val"]].round(1)
overview

mae     rmse  n_train  n_val
target        horizon_days                                 
pv            1             1811.0   3688.4    41982  17520
              7             2353.5   4940.9    41982  17520
              14            2321.4   4796.9    41982  17520
wind_onshore  1             6761.7   8873.4    41982  17520
              7             7706.4  10146.0    41982  17520
              14            7965.6  10475.9    41982  17520
wind_offshore 1             1487.1   1785.4    41982  17520
              7             1546.8   1849.5    41982  17520
              14            1545.6   1860.0    41982  17520
load          1             1652.5   2347.4    41886  17520
              7             1941.7   2735.2    41862  17520
              14            2115.9   2974.8    41862  17520

## Validation MAE by horizon

```{figure} ../../output/images/05_lgbm_mae_by_horizon.png
:name: fig-05-lgbm-mae-by-horizon
LightGBM validation MAE by forecast horizon, one panel per target.
```

In [5]:
fig = plot_mae_by_horizon(results, color=COLOR)
fig.suptitle(f"{MODEL_LABEL} — validation MAE by forecast horizon", y=1.04)
fig.savefig(paths.images_path / "05_lgbm_mae_by_horizon.png", dpi=150, bbox_inches="tight")

## Actual vs. predicted — summer window

```{figure} ../../output/images/05_lgbm_actual_vs_predicted_summer.png
:name: fig-05-lgbm-actual-vs-predicted-summer
Actual (black) vs. LightGBM (green), 2024-07-01 .. 2024-07-21.
```

In [6]:
fig = plot_actual_vs_predicted(val_predictions, EXAMPLE_WINDOWS["summer"], color=COLOR, model_label=MODEL_LABEL)
fig.suptitle(f"{MODEL_LABEL} — actual vs. predicted, summer window", y=1.04)
fig.savefig(paths.images_path / "05_lgbm_actual_vs_predicted_summer.png", dpi=150, bbox_inches="tight")

## Actual vs. predicted — winter window

```{figure} ../../output/images/05_lgbm_actual_vs_predicted_winter.png
:name: fig-05-lgbm-actual-vs-predicted-winter
Actual (black) vs. LightGBM (green), 2025-01-01 .. 2025-01-21.
```

In [7]:
fig = plot_actual_vs_predicted(val_predictions, EXAMPLE_WINDOWS["winter"], color=COLOR, model_label=MODEL_LABEL)
fig.suptitle(f"{MODEL_LABEL} — actual vs. predicted, winter window", y=1.04)
fig.savefig(paths.images_path / "05_lgbm_actual_vs_predicted_winter.png", dpi=150, bbox_inches="tight")

## Error analysis — full validation period

7-day rolling mean absolute error over the full ~2-year validation
period, one line per horizon. Reveals whether error is roughly constant
over time (homoskedastic) or clusters seasonally / grows with some
regime — e.g. PV error should track the summer/winter irradiance swing.

```{figure} ../../output/images/05_lgbm_rolling_error.png
:name: fig-05-lgbm-rolling-error
7-day rolling mean absolute error over the full validation period, by
target and horizon.
```

In [8]:
fig = plot_rolling_error(val_predictions)
fig.suptitle(f"{MODEL_LABEL} — 7-day rolling validation MAE", y=1.06)
fig.savefig(paths.images_path / "05_lgbm_rolling_error.png", dpi=150, bbox_inches="tight")

## Feature importance

LightGBM gain-based importance, normalized to a share of the total per
(target, horizon) fit, so columns are directly comparable.

```{figure} ../../output/images/05_lgbm_feature_importance.png
:name: fig-05-lgbm-feature-importance
Feature importance share (gain) by target and horizon.
```

In [9]:
fig = plot_importance_heatmaps(importances)
fig.suptitle(f"{MODEL_LABEL} — feature importance", y=1.02)
fig.savefig(paths.images_path / "05_lgbm_feature_importance.png", dpi=150, bbox_inches="tight")